# Media de pernoctaciones — EOH (INE)
  
**Ticket:** 3.3. Pernoctaciones por ciudad — zoom Nacional (CCAA) vs Internacional  
**Fuente externa:** Encuesta de Ocupación Hotelera (EOH), INE  
**Dataset:** `clean_dataset_INE_EOH_20_07_2026.csv` (salida de Data Cleaning)

Este notebook responde a una parte de la pregunta de negocio:

> ¿Es necesario ajustar nuestras ofertas al perfil del viajero y a la demanda de
> pernoctaciones en las ciudades donde estamos presentes, considerando las cifras
> oficiales sobre procedencia, meses de visita y media de pernoctaciones por
> comunidad autónoma?

En concreto, aquí se calcula **cuántas noches se queda de media un viajero** en cada
uno de los 8 mercados donde operamos, separando **viajero nacional** (residente en
España) y **viajero internacional** (residente en el extranjero), y haciendo un
**zoom** sobre las comunidades autónomas de origen del viajero nacional.

**Aclaración de alcance:** la EOH mide establecimientos hoteleros (hotel, hostal,
pensión, etc.), no apartamentos turísticos. Por tanto la estancia media se usa como
**aproximación (proxy) del comportamiento del viajero por territorio**, no como una
medida directa del segmento de apartamentos.

## 1. Librerías

In [1]:
# Librerías básicas para el análisis
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo sencillo y homogéneo con el resto de notebooks del equipo
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

## 2. Parámetros y carga de datos

Igual que en el resto de notebooks, **solo hay que tocar esta sección** cuando llegue
un fichero nuevo del INE: se actualiza el nombre del dataset limpio y, si hace falta,
el periodo de análisis.

La ruta se resuelve buscando la carpeta raíz del proyecto (`Equip_34`), de forma que
funcione en el ordenador de cualquier compañero sin escribir rutas locales.

In [2]:
# --- Únicas líneas a cambiar cuando llegue un dataset nuevo ---
DATASET_LIMPIO = "clean_dataset_INE_EOH_20_07_2026.csv"

# Periodo de análisis: 3 años completos (2023-2025). Se deja fuera 2026 por estar
# en curso (incompleto). Mismo criterio que el EDA, para que los números coincidan.
PERIODO_INICIO = pd.Timestamp("2023-01-01")
PERIODO_FIN = pd.Timestamp("2025-12-01")


def encontrar_raiz_proyecto(nombre_carpeta="Equip_34"):
    """Busca la carpeta raíz del proyecto subiendo desde el directorio actual."""
    actual = Path.cwd()

    for carpeta in [actual] + list(actual.parents):
        if carpeta.name == nombre_carpeta:
            return carpeta

    raise FileNotFoundError(
        f"No se encontró la carpeta '{nombre_carpeta}' subiendo desde {actual}"
    )


raiz_proyecto = encontrar_raiz_proyecto("Equip_34")

# El dataset limpio se guarda junto al resto de datos, en la carpeta Data.
# Este notebook vive en Scripts (misma estructura que los demás del proyecto).
ruta_csv = raiz_proyecto / "Data" / DATASET_LIMPIO
if not ruta_csv.exists():
    raise FileNotFoundError(f"No se encontró el dataset limpio: {DATASET_LIMPIO}")

# El Data Cleaning exporta con ';' de separador, coma decimal y BOM utf-8.
eoh = pd.read_csv(ruta_csv, sep=";", decimal=",", encoding="utf-8-sig")

# 'fecha' viene como texto; se convierte a fecha real para poder filtrar por periodo.
eoh["fecha"] = pd.to_datetime(eoh["fecha"])

print(f"Filas cargadas: {len(eoh):,}")
print(f"Columnas: {list(eoh.columns)}")

FileNotFoundError: No se encontró el dataset limpio: clean_dataset_INE_EOH_20_07_2026.csv

## 3. Correspondencia entre nuestras ciudades y la geografía del INE

StaySpain opera en 8 mercados. La EOH no ofrece todos al mismo nivel geográfico, así
que cada mercado se localiza en la tabla que mejor lo cubre (misma decisión que el EDA):

- **5 ciudades** (Barcelona, Madrid, València, Málaga, Sevilla) → tabla **2078**
  (puntos turísticos = municipio).
- **3 provincias/zonas** (Mallorca, Menorca, Girona) → tabla **2039**
  (zonas turísticas), porque a nivel de municipio no están disponibles.

Para el zoom del viajero nacional por comunidad autónoma de origen se usa la tabla
**2074** (valores absolutos por CCAA de destino y procedencia).

El cruce se hace por `geo_key` (nombre en minúsculas y sin tildes), que es la clave
estable que crea el Data Cleaning.

In [ ]:
# Mercados de StaySpain y su geografía en la EOH.
# La clave de cruce es geo_key (minúsculas, sin tildes), tal como la genera el
# Data Cleaning. Se anota entre paréntesis el nombre original solo como referencia.

# 5 ciudades que están en la tabla 2078 (puntos turísticos)
CIUDADES_2078 = {
    "Barcelona": "barcelona",
    "Madrid": "madrid",
    "Valencia": "valencia",
    "Málaga": "malaga",
    "Sevilla": "sevilla",
}

# 3 mercados que están en la tabla 2039 (zonas turísticas)
ZONAS_2039 = {
    "Mallorca": "baleares (illes): isla de mallorca",
    "Menorca": "baleares (illes): isla de menorca",
    "Girona": "cataluna: costa brava",
}

# Comunidad autónoma de cada mercado en la tabla 2074 (para el zoom nacional)
CCAA_2074 = {
    "Barcelona": "cataluna",
    "Girona": "cataluna",
    "Madrid": "madrid, comunidad de",
    "Valencia": "comunitat valenciana",
    "Málaga": "andalucia",
    "Sevilla": "andalucia",
    "Mallorca": "balears, illes",
    "Menorca": "balears, illes",
}

# Orden fijo de los mercados en todas las tablas y gráficos (de más a menos conocido)
ORDEN_MERCADOS = ["Barcelona", "Madrid", "Valencia", "Málaga", "Sevilla",
                  "Mallorca", "Menorca", "Girona"]

## 4. Cómo se calcula la media de pernoctaciones

La estancia media **no es una columna del dataset**: se reconstruye dividiendo el
total de pernoctaciones (noches) entre el total de viajeros (personas):

$$\text{estancia media} = \frac{\text{pernoctaciones}}{\text{viajeros}}$$

**Punto clave (no equivocarse aquí):** no se puede hacer la media de las estancias
medias mensuales. Un mes flojo pesaría igual que uno de temporada alta y falsearía el
resultado. Lo correcto es **sumar primero** todas las pernoctaciones y todos los
viajeros del periodo, y **dividir después**.

Además se excluyen siempre las celdas marcadas por el INE como `sin_dato` (secreto
estadístico o sin muestra ese mes): son NaN, **no ceros**, y no deben entrar en la suma.

In [ ]:
def media_pernoctaciones(df):
    """Estancia media = suma de pernoctaciones / suma de viajeros.

    Recibe un dataframe ya filtrado (por geografía, procedencia, periodo...) que
    contiene las dos métricas ('pernoctaciones' y 'viajeros') en la columna 'valor'.
    Suma cada métrica por separado y divide. Devuelve las noches medias (float).

    Se apoya en que las filas 'sin_dato' ya vienen como NaN en 'valor', de modo que
    .sum() las ignora automáticamente.
    """
    pernoctaciones = df.loc[df["metrica"] == "pernoctaciones", "valor"].sum()
    viajeros = df.loc[df["metrica"] == "viajeros", "valor"].sum()

    # Si no hay viajeros (todo el bloque era secreto estadístico), no se puede dividir
    if viajeros == 0:
        return float("nan")

    return pernoctaciones / viajeros

## 5. Preparación del subconjunto de trabajo

Se filtra el dataset al periodo 2023-2025 y a las dos métricas que necesitamos
(`pernoctaciones` y `viajeros`). La tabla 2069 se queda fuera de forma natural porque
está en porcentajes, no en conteos, y aquí solo se trabaja con valores absolutos.

In [ ]:
# Nos quedamos solo con las filas útiles para este ticket:
# - periodo 2023-2025
# - métricas de conteo (pernoctaciones y viajeros)
# La 2069 (porcentajes) no entra porque su 'metrica'/'unidad' no es de conteo.
base = eoh[
    eoh["fecha"].between(PERIODO_INICIO, PERIODO_FIN)
    & eoh["metrica"].isin(["pernoctaciones", "viajeros"])
].copy()

print(f"Filas en el periodo y métricas de interés: {len(base):,}")
print("\nCategorías de procedencia disponibles:")
print(base["categoria"].value_counts(dropna=False))

En la columna `categoria` la procedencia aparece con las etiquetas del INE. Se
definen abajo qué valor corresponde al viajero **nacional** y cuál al
**internacional**, para no depender de cómo estén escritas exactamente.

In [ ]:
# Etiquetas del INE para la procedencia (residencia del viajero).
# Se comprueban contra el value_counts de arriba antes de usarlas.
NACIONAL = "Residentes en España"
INTERNACIONAL = "Residentes en el Extranjero"

# Aviso claro si alguna etiqueta no aparece tal cual (por si cambia en el futuro)
for etiqueta in (NACIONAL, INTERNACIONAL):
    if etiqueta not in base["categoria"].unique():
        print(f"AVISO: no se encuentra la categoría '{etiqueta}'. "
              f"Revisar las etiquetas de 'categoria' en el value_counts anterior.")

## 6. Media de pernoctaciones por ciudad (total)

Primer nivel de lectura del ticket: cuántas noches se queda de media el viajero en
cada mercado, **sin separar todavía** por procedencia. Se recorre cada mercado, se
localiza en su tabla (2078 o 2039) y se aplica la función de estancia media.

In [ ]:
def filtrar_mercado(df, mercado):
    """Devuelve las filas del dataset correspondientes a un mercado de StaySpain.

    Elige la tabla según dónde esté disponible el mercado:
    - 5 ciudades -> tabla 2078 (puntos turísticos)
    - Mallorca, Menorca, Girona -> tabla 2039 (zonas turísticas)
    El cruce se hace por geo_key (clave sin tildes del Data Cleaning).
    """
    if mercado in CIUDADES_2078:
        geo_key = CIUDADES_2078[mercado]
        return df[(df["tabla"] == 2078) & (df["geo_key"] == geo_key)]
    else:
        geo_key = ZONAS_2039[mercado]
        return df[(df["tabla"] == 2039) & (df["geo_key"] == geo_key)]


# Media total por mercado (todas las procedencias juntas)
filas_total = []
for mercado in ORDEN_MERCADOS:
    datos_mercado = filtrar_mercado(base, mercado)
    filas_total.append({
        "mercado": mercado,
        "estancia_media": media_pernoctaciones(datos_mercado),
    })

estancia_total = pd.DataFrame(filas_total)
estancia_total["estancia_media"] = estancia_total["estancia_media"].round(2)
estancia_total

In [ ]:
# Gráfico: media de pernoctaciones por mercado (total)
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=estancia_total, x="estancia_media", y="mercado",
            order=ORDEN_MERCADOS, color="#4C72B0", ax=ax)

ax.set_xlabel("Noches de media por viajero (2023-2025)")
ax.set_ylabel("")
ax.set_title("Media de pernoctaciones por ciudad")

# Etiqueta con el valor al final de cada barra, para leerlo sin mirar el eje
for i, fila in estancia_total.iterrows():
    ax.text(fila["estancia_media"] + 0.03, i, f"{fila['estancia_media']:.2f}",
            va="center")

plt.tight_layout()
plt.show()

## 7. Zoom: viajero nacional vs internacional

Segundo nivel de lectura (el zoom que pide el ticket). Para cada mercado se calcula la
estancia media **por separado** según la procedencia del viajero. Así se ve si el
viajero internacional se queda más noches que el nacional en cada ciudad.

In [ ]:
# Media por mercado separando nacional / internacional
filas_zoom = []
for mercado in ORDEN_MERCADOS:
    datos_mercado = filtrar_mercado(base, mercado)

    nacional = datos_mercado[datos_mercado["categoria"] == NACIONAL]
    internacional = datos_mercado[datos_mercado["categoria"] == INTERNACIONAL]

    filas_zoom.append({
        "mercado": mercado,
        "nacional": media_pernoctaciones(nacional),
        "internacional": media_pernoctaciones(internacional),
    })

estancia_zoom = pd.DataFrame(filas_zoom).round(2)
estancia_zoom

In [ ]:
# Gráfico de barras agrupadas: nacional vs internacional por mercado.
# Se pasa a formato largo para que seaborn dibuje las dos barras por ciudad.
zoom_largo = estancia_zoom.melt(
    id_vars="mercado", value_vars=["nacional", "internacional"],
    var_name="procedencia", value_name="estancia_media"
)

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(data=zoom_largo, x="mercado", y="estancia_media",
            hue="procedencia", order=ORDEN_MERCADOS,
            palette={"nacional": "#4C72B0", "internacional": "#DD8452"}, ax=ax)

ax.set_xlabel("")
ax.set_ylabel("Noches de media por viajero (2023-2025)")
ax.set_title("Media de pernoctaciones por ciudad: nacional vs internacional")
ax.legend(title="Procedencia")

plt.tight_layout()
plt.show()

## 8. Zoom del viajero nacional: ¿de qué comunidades autónomas viene?

El ticket pide, dentro del viajero nacional, bajar a nivel de **comunidad autónoma de
origen — Nacional(CCAA)**. Este detalle no está en las tablas 2078/2039, así que se usa
la tabla **2074**, que sí distingue la CCAA de destino y la procedencia.

Aquí se calcula, para la comunidad autónoma de cada mercado, la estancia media del
viajero nacional y la del internacional, como contexto del dato de ciudad. Es un
apoyo al zoom: las tablas de ciudad dan el "cuánto"; la 2074 ayuda a situarlo dentro
del comportamiento de la comunidad autónoma.

In [ ]:
# Estancia media a nivel de comunidad autónoma (tabla 2074), por procedencia.
# Sirve de contexto para el viajero nacional de cada mercado.
filas_ccaa = []
mercados_por_ccaa = {}  # para no repetir el cálculo en CCAA compartidas
for mercado in ORDEN_MERCADOS:
    ccaa_key = CCAA_2074[mercado]
    mercados_por_ccaa.setdefault(ccaa_key, []).append(mercado)

for ccaa_key, mercados in mercados_por_ccaa.items():
    datos_ccaa = base[(base["tabla"] == 2074)
                      & (base["nivel_geo"] == "ccaa")
                      & (base["geo_key"] == ccaa_key)]

    nacional = datos_ccaa[datos_ccaa["categoria"] == NACIONAL]
    internacional = datos_ccaa[datos_ccaa["categoria"] == INTERNACIONAL]

    filas_ccaa.append({
        "comunidad_autonoma": ccaa_key,
        "mercados": ", ".join(mercados),
        "nacional": media_pernoctaciones(nacional),
        "internacional": media_pernoctaciones(internacional),
    })

estancia_ccaa = pd.DataFrame(filas_ccaa).round(2)
estancia_ccaa

## 9. Resumen del ticket

Este notebook deja calculados, para los 8 mercados de StaySpain y el periodo 2023-2025:

1. La **media de pernoctaciones total** por ciudad (sección 6).
2. El **zoom nacional vs internacional** por ciudad (sección 7).
3. El **contexto por comunidad autónoma** de origen del viajero nacional (sección 8).

Estos resultados alimentan el bloque **3.3. Pernoctaciones por ciudad** de la
presentación conjunta, y se compararán después con la oferta de StaySpain (estancias
mínimas, tipo de alojamiento) para valorar si conviene ajustarla al comportamiento
real del viajero en cada mercado.

**Limitaciones a recordar al presentar:**
- La EOH es hotelera, no de apartamentos: el dato es un **proxy** del viajero, no de
  nuestro segmento.
- Los datos de junio de 2025 en adelante son **provisionales** (columna `provisional`).
- Mallorca, Menorca y Girona se miden a nivel de zona turística/provincia, no de
  municipio, por lo que no son estrictamente comparables con las 5 ciudades.